# Sample Modeling Table Test

Testing if the loader design can work with a small table that resembles the final processed dataset.

## The dataset is expected to include
customer_id
treatment
offer_id
treatment_time
outcome
age
income
gender
membership_duration
previous_transactions
previous_spending
previous_offers_received
previous_offers_viewed

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SyntheticUpliftDataset(Dataset):
    def __init__(
        self,
        customer_features,
        incentive_features,
        treatment,
        outcome
    ):
        self.customer_features = customer_features
        self.incentive_features = incentive_features
        self.treatment = treatment
        self.outcome = outcome

    # To know how many samples the dataset include
    def __len__(self):
        return self.customer_features.shape[0]

    # Returns a single dataset based on the index used for single customer and treatment decision
    def __getitem__(self, index):
        return {
            "customer_features": self.customer_features[index],
            "incentive_features": self.incentive_features[index],
            "treatment": self.treatment[index],
            "outcome": self.outcome[index]
        }

In [14]:
import pandas as pd

# Sample fake data table matching the expected feature set. 
sample_table = pd.DataFrame({
    "customer_id": [
        "C001", "C002", "C003", "C004", "C005",
        "C006", "C007", "C008", "C009", "C010"
    ],
    "treatment": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    "offer_id": [
        "control",
        "offer_1", "offer_2", "offer_3", "offer_4",
        "offer_5", "offer_6", "offer_7", "offer_8",
        "offer_9"
    ],
    "treatment_time": [0, 24, 48, 72, 96, 120, 144, 168, 192, 216],
    "outcome": [0, 1, 0, 1, 1, 0, 1, 0, 1, 0],
    "age": [25, 31, 42, 55, 28, 64, 37, 49, 71, 33],
    "income": [
        45000, 52000, 68000, 75000, 48000,
        90000, 61000, 70000, 82000, 57000
    ],
    "gender": [
        "M", "F", "M", "F", "M",
        "F", "M", "F", "M", "F"
    ],
    "membership_duration": [12, 24, 36, 48, 60, 18, 30, 42, 54, 66],
    "previous_transactions": [2, 5, 8, 10, 3, 12, 6, 9, 15, 4],
    "previous_spending": [
        25.50, 80.00, 145.75, 210.20, 45.00,
        325.50, 110.00, 178.25, 410.00, 65.75
    ],
    "previous_offers_received": [1, 2, 3, 4, 1, 5, 2, 3, 6, 1],
    "previous_offers_viewed": [1, 2, 2, 3, 1, 4, 2, 3, 5, 1]
})

## External inclusion validation
sample_table.loc[len(sample_table)] = [
    "C011",
    10,
    "offer_10",
    240,
    1,
    46,
    63000,
    "M",
    20,
    7,
    130.50,
    3,
    2
]

# Validating the created fake data
print("Table shape:", sample_table.shape)
print("Columns:", sample_table.columns.tolist())
print(sample_table.head())
print("Treatments:", sorted(sample_table["treatment"].unique()))
print("Outcomes:", sorted(sample_table["outcome"].unique()))

Table shape: (11, 13)
Columns: ['customer_id', 'treatment', 'offer_id', 'treatment_time', 'outcome', 'age', 'income', 'gender', 'membership_duration', 'previous_transactions', 'previous_spending', 'previous_offers_received', 'previous_offers_viewed']
  customer_id  treatment offer_id  treatment_time  outcome  age  income  \
0        C001          0  control               0        0   25   45000   
1        C002          1  offer_1              24        1   31   52000   
2        C003          2  offer_2              48        0   42   68000   
3        C004          3  offer_3              72        1   55   75000   
4        C005          4  offer_4              96        1   28   48000   

  gender  membership_duration  previous_transactions  previous_spending  \
0      M                   12                      2              25.50   
1      F                   24                      5              80.00   
2      M                   36                      8             145.75  

In [15]:
# Adding offer or incentive features

sample_table["offer_type_code"] = [
    0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3
]

sample_table["reward"] = [
    0, 5, 10, 15, 20, 5, 10, 15, 20, 0, 0
]

sample_table["difficulty"] = [
    0, 5, 10, 15, 20, 5, 10, 15, 20, 0, 0
]

sample_table["duration"] = [
    0, 72, 72, 120, 120, 72, 72, 120, 120, 72, 120
]

sample_table["channel_count"] = [
    0, 2, 3, 2, 4, 2, 3, 2, 4, 3, 4
]

sample_table["is_informational"] = [
    0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1
]

# Validation after adding offer features
print(sample_table.shape)
print(sample_table.columns.tolist())
print(sample_table.head())

(11, 19)
['customer_id', 'treatment', 'offer_id', 'treatment_time', 'outcome', 'age', 'income', 'gender', 'membership_duration', 'previous_transactions', 'previous_spending', 'previous_offers_received', 'previous_offers_viewed', 'offer_type_code', 'reward', 'difficulty', 'duration', 'channel_count', 'is_informational']
  customer_id  treatment offer_id  treatment_time  outcome  age  income  \
0        C001          0  control               0        0   25   45000   
1        C002          1  offer_1              24        1   31   52000   
2        C003          2  offer_2              48        0   42   68000   
3        C004          3  offer_3              72        1   55   75000   
4        C005          4  offer_4              96        1   28   48000   

  gender  membership_duration  previous_transactions  previous_spending  \
0      M                   12                      2              25.50   
1      F                   24                      5              80.00   
2  

In [16]:
import torch

# Breaking the dataset into independant tensors.

# Start with encoding gender 
sample_table["gender_code"] = sample_table["gender"].map({
    "M": 0,
    "F": 1
})

# customer feature set tensor
customer_features_tensor = torch.tensor(
    sample_table[
        [
            "age",
            "income",
            "gender_code",
            "membership_duration",
            "previous_transactions",
            "previous_spending",
            "previous_offers_received",
            "previous_offers_viewed"
        ]
    ].to_numpy(),
    dtype=torch.float32
)

incentive_features_tensor = torch.tensor(
    sample_table[
        [
            "offer_type_code",
            "reward",
            "difficulty",
            "duration",
            "channel_count",
            "is_informational"
        ]
    ].to_numpy(),
    dtype=torch.float32
)

treatment_tensor = torch.tensor(
    sample_table["treatment"].to_numpy(),
    dtype=torch.long
)

# Reshape to match the planned outcome shape
outcome_tensor = torch.tensor(
    sample_table["outcome"].to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)


print("Customer features tensor:", customer_features_tensor.shape)
print("Incentive features tensor:", incentive_features_tensor.shape)
print("Treatment tensor:", treatment_tensor.shape)
print("Outcome tensor:", outcome_tensor.shape)



table_dataset = SyntheticUpliftDataset(
    customer_features=customer_features_tensor,
    incentive_features=incentive_features_tensor,
    treatment=treatment_tensor,
    outcome=outcome_tensor
)

print("Artificial dataset size:", len(table_dataset))

Customer features tensor: torch.Size([11, 8])
Incentive features tensor: torch.Size([11, 6])
Treatment tensor: torch.Size([11])
Outcome tensor: torch.Size([11, 1])
Artificial dataset size: 11


In [ ]:
table_loader = DataLoader(
    table_dataset,
    batch_size=4,
    shuffle=False
)